In [1]:
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split, KFold
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import  RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor



from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

from pathlib import Path



from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    log_loss,
    brier_score_loss,
)

from scipy import stats

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(60)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

SEED = 42
CV_SAMPLE_SIZE = 200_000

PRED_DIR = Path("../data/processed")
PRED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
Xtr = pl.read_parquet("../data/processed/X_train_pd.parquet")

TARGET = "y"

LEAKAGE_VARS = ['grade']

PERFORMANCE_VARS = [
    'remaining_princ_for_tot_amnt_fund',
    'paym_rec_for_tot_amnt_fund',
    'princ_rec',
    'interest_rec',
    'late_fees_rec'
]

CAT  = [c for c, dt in Xtr.schema.items() if dt == pl.String and c != 'grade']
NUM  = [c for c, dt in Xtr.schema.items() if dt.is_numeric()]

LOG_VARS   = [c for c in NUM if c.startswith("log_")]    # 3 logs creados en 02
FLAG_VARS  = [c for c in NUM if c.startswith("flag_")]   # 5 flags de imputación
NUM_BASE   = [c for c in NUM if c not in LOG_VARS + FLAG_VARS + PERFORMANCE_VARS ]  # 29 originales + 3 variables creadas en EDA
RAW_OF_LOG = [c.removeprefix("log_") for c in LOG_VARS]  # variables que tienen su log
VIF_DROP   = ["monthly_payment"]                         # VIF > 10 (colineal con funded_amnt)

ING_VARIABLES_BASICAS = [
    'antiguedad_crediticia_meses',
    'ratio_financiacion_ingreso',
    'emp_length_ordinal'
]
VIF_DROP_ING = [
    "earliest_cr_line_year",
    "monthly_payment",
    "issue_date_year"]

PD = ['pd_lgbm']

# VARS = NUM + CAT

X_train = Xtr.to_pandas()
X_test  = pl.read_parquet("../data/processed/X_test_pd.parquet").to_pandas()
y_train = pl.read_parquet("../data/processed/y_train.parquet")["y"].to_numpy()
y_test  = pl.read_parquet("../data/processed/y_test.parquet")["y"].to_numpy()

print(f"num_base={len(NUM_BASE)} | logs={len(LOG_VARS)} | flags={len(FLAG_VARS)} | cat={len(CAT)}")

num_base=33 | logs=3 | flags=5 | cat=10


In [3]:
# ¿Están las variables de desempeño en X_train / X_test?
faltan = [c for c in PERFORMANCE_VARS if c not in X_train.columns]
print("Faltan en X_train:", faltan)
print(X_train.shape, X_test.shape)
# print("Índice único:", X_train.index.is_unique, X_test.index.is_unique)

Faltan en X_train: []
(1711714, 57) (427929, 57)


## EDA

### Descripción de las variables de comportamiento   
1. `remaining_princ_for_tot_amnt_fund`:   Principal que permanece pendiente.
2. `paym_rec_for_tot_amnt_fund` :  	 pagos recibidos respecto al monto financiado.
3. `princ_rec` : Principal recuperado.
4. `interest_rec`: Intereses recuperados.
5. `late_fees_rec` : Penalizaciones o cargos por mora recuperados.
6.  `funded_amnt`: Monto financiado del préstamo


### Expected Credit Loss

PD × LGD × EAD = ECL

- PD : Probability of default - ¿Cuál es la probabilidad de que el préstamo termine en impago?

- LGD: Loss Given Default - Si el préstamo impaga, ¿qué proporción de la exposición se pierde después de las recuperaciones?

- EAD: Exposure at default - ¿Cuánto dinero queda expuesto cuando ocurre el impago?


In [4]:
# Analisis de los prestamos impagos solo train
D = X_train.loc[y_train == 1, ["funded_amnt"] + PERFORMANCE_VARS].copy()

print(f"Impagos en train: {len(D)}")

# Diagnóstico clave: ¿cuántos impagados tienen principal pendiente = 0?
print("% con principal pendiente = 0 ->", (D["remaining_princ_for_tot_amnt_fund"] == 0).mean().round(3) )

D.describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2)

Impagos en train: 223773
% con principal pendiente = 0 -> 0.918


,count,mean,std,min,1%,25%,50%,75%,99%,max
funded_amnt,223773.0,15514.03,8813.82,500.0,2000.00,9000.00,14125.00,20300.00,35000.00,40000.00
remaining_princ_for_tot_amnt_fund,223773.0,880.75,3763.44,0.0,0.00,0.00,0.00,0.00,21344.32,40000.00
paym_rec_for_tot_amnt_fund,223773.0,8313.05,6759.85,0.0,336.31,3470.80,6492.22,11199.46,31964.82,61947.53
princ_rec,223773.0,4498.22,4230.88,0.0,67.85,1557.80,3231.11,6052.66,20087.26,37600.00
interest_rec,223773.0,2731.32,2783.73,0.0,39.64,862.73,1813.37,3634.42,13349.56,26915.29
late_fees_rec,223773.0,5.25,20.19,-0.0,0.00,0.00,0.00,0.00,82.24,1427.25


### Hipótesis A

En esta hipótesis comprobaremos todo lo recibido como recuperación. EL punto es hayar la `perdida_obs`, ya que es la variable que nos falta. Aplicando la siguiente formula.

$$[LGD_A=\frac{\text{Pérdida observada}}{EAD}]$$

Resultado de la Hipótesis A:
- EAD media: 11,016 (exposición estimada al impago).
- Pérdida media: 7,201.
- LGD media: 60.9% (se pierde aproximadamente el 61% de la exposición).
- LGD = 0: 7.3% de los préstamos.
- LGD = 1: 0.3% de los préstamos.

Conclusión: la pérdida estimada es del 61% de la exposición, pero debemos validar si esta fórmula representa correctamente la pérdida real.

In [5]:
# Hipótesis de trabajo (a validar):
#   exposición al impago = prestado - principal ya cobrado
#   pérdida              = prestado - total cobrado (principal + intereses + comisiones)
#   LGD                  = pérdida / exposición, recortada a [0, 1]
D["Ead_obs"] = D["funded_amnt"] - D["princ_rec"]
D["perdida_obs"] = D["funded_amnt"] - D["paym_rec_for_tot_amnt_fund"]
D["Lgd_obs"] = (D["perdida_obs"] / D["Ead_obs"]).clip(0, 1)

print(D[["Ead_obs", "perdida_obs", "Lgd_obs"]].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(3))
print("% LGD = 1:", (D["Lgd_obs"] >= 0.999).mean().round(3),
      "| % LGD = 0:", (D["Lgd_obs"] <= 0.001).mean().round(3))

                count       mean       std       min        1%       25%  \
Ead_obs      223773.0  11015.815  7719.151      0.00   591.529  4889.440   
perdida_obs  223773.0   7200.983  6721.532 -26947.53 -4064.700  2415.420   
Lgd_obs      223773.0      0.609     0.272      0.00     0.000     0.467   

                  50%        75%        99%      max  
Ead_obs      9350.340  15381.330  32995.700  40000.0  
perdida_obs  5722.160  10550.030  28283.416  40000.0  
Lgd_obs         0.685      0.809      0.981      1.0  
% LGD = 1: 0.003 | % LGD = 0: 0.073


### Hipótesis B

En esta hipótesis solo contaremos el capital devueto. EL punto es hayar la `rec_impl`, ya que es la variable que nos falta. Aplicando la siguiente formula.

$$[LGD_B=1-\frac{\text{Recuperaciones}}{EAD}]$$

El objetivo de esta hipótesis es buscar `rec_impl` valores negativos importantes.

- Los resultados nos indica que ningun prestamos tiene `rec_impl` menor que -1. 
- El 38.3% de los préstamos tiene rec_impl aproximadamente igual a cero. Es decir, sus pagos quedan prácticamente explicados por capital, intereses y comisiones.

In [6]:
B = X_train.loc[y_train == 1, ["funded_amnt"] + PERFORMANCE_VARS].copy()

# Recuperaciones post-impago implícitas (lo que no explican principal, intereses ni comisiones)
B["rec_impl"] = (
    B["paym_rec_for_tot_amnt_fund"] - B["princ_rec"] - B["interest_rec"] - B["late_fees_rec"])

print(B["rec_impl"].describe(percentiles=[.01, .25, .5, .75, .99]).round(2))
print("\n % rec_impl < -1 (no cuadra): ", (B["rec_impl"] < -1).mean().round(4))
print("\n % sin recuperación (≈ 0):  ", (B["rec_impl"].abs() <= 1).mean().round(3))

count    223773.00
mean       1078.27
std        1774.38
min          -0.01
1%           -0.00
25%           0.00
50%         403.66
75%        1545.80
99%        7995.17
max       39859.55
Name: rec_impl, dtype: float64

 % rec_impl < -1 (no cuadra):  0.0

 % sin recuperación (≈ 0):   0.383


Si EAD es cero, no podemos dividir entre cero.

In [7]:
# EAD reconstruido. Si EAD ≈ 0, el LGD es indefinido (NaN), no 1.
B["Ead_obs"] = B["funded_amnt"] - B["princ_rec"]
print("% EAD <= 1: ", (B["Ead_obs"] <= 1).mean().round(4))

#Cuando EAD es 1 o menos, asigna NaN (valor faltante), no una LGD de 100%.
ead = B["Ead_obs"].where(B["Ead_obs"] > 1)

# ¿Qué pasa con el 8,2% (18376) que sí tiene principal pendiente?
m = B["remaining_princ_for_tot_amnt_fund"] > 0 
print(B.loc[m, ["Ead_obs", "remaining_princ_for_tot_amnt_fund"]].describe().round(0))

% EAD <= 1:  0.0003
       Ead_obs  remaining_princ_for_tot_amnt_fund
count  18376.0                            18376.0
mean   10725.0                            10725.0
std     8179.0                             8179.0
min        0.0                                0.0
25%     4343.0                             4343.0
50%     8768.0                             8768.0
75%    15056.0                            15056.0
max    40000.0                            40000.0


In [8]:
# Las dos definiciones de LGD
B["lgd_A"] = ((B["funded_amnt"] - B["paym_rec_for_tot_amnt_fund"]) / ead).clip(0, 1)
B["lgd_B"] = (1 - B["rec_impl"].clip(lower=0) / ead).clip(0, 1)

resultados = []

for nombre in ["lgd_A", "lgd_B"]:
    ok = B[nombre].notna()
    lgd = B.loc[ok, nombre]
    ead_ok = ead[ok]

    resultados.append({
        "Modelo": nombre.upper(),
        "N préstamos": int(ok.sum()),
        "LGD media (%)": lgd.mean() * 100,
        "LGD ponderada (%)": np.average(lgd, weights=ead_ok) * 100,
        "% LGD = 100%": (lgd >= 0.999).mean() * 100,
        "% LGD = 0%": (lgd <= 0.001).mean() * 100,
        "Pérdida total": (lgd * ead_ok).sum()
    })

tabla_lgd = pd.DataFrame(resultados).set_index("Modelo")

display(
    tabla_lgd.style
    .format({
        "N préstamos": "{:,.0f}",
        "LGD media (%)": "{:.2f}%",
        "LGD ponderada (%)": "{:.2f}%",
        "% LGD = 100%": "{:.2f}%",
        "% LGD = 0%": "{:.2f}%",
        "Pérdida total": "{:,.0f}"
    })
    .background_gradient(subset=["LGD media (%)", "LGD ponderada (%)"])
    .set_caption("Comparación de hipótesis de LGD")
)

,N préstamos,LGD media (%),LGD ponderada (%),% LGD = 100%,% LGD = 0%,Pérdida total
Modelo,,,,,,
LGD_A,"223,695",60.89%,66.68%,0.34%,7.31%,"1,643,763,827"
LGD_B,"223,695",90.17%,90.23%,39.72%,0.35%,"2,224,127,280"


### Definición de los objetivos observados (solo impagados del train)

**Exposición (EAD).** `EAD = funded_amnt − princ_rec`. Validado: coincide con el principal pendiente en el 100% de los impagados que lo conservan; en el 91,8% restante el saldo fue dado de baja.

**Recuperación post-impago.** `rec = paym_rec − princ_rec − interest_rec − late_fees_rec`. Es lo cobrado que no es principal, interés ni comisión previos al impago. Nunca es negativa; 38,3% de los impagados no recupera nada (mediana de los que sí: ≈ 400).

**LGD.** `LGD = 1 − rec / EAD`, en [0, 1]. Media 0,90; 39,7% de los impagados con pérdida total.

**Por qué esta definición y no restar todo lo cobrado.** Los intereses cobrados antes del impago eran ingreso, no recuperación del principal perdido. Restarlos baja el LGD medio a 0,61 y reduce la pérdida total en 580 M (−26%), una estimación anti-conservadora para un cálculo de provisiones.

**Exclusión.** 0,03% de impagados con EAD ≤ 1 (LGD indefinido) se excluyen del entrenamiento de LGD.


### Construimos las variables para el modelo

In [9]:
# Solo impagados del TRAIN. Se conserva el índice para unir con PD y con el resto.
T = X_train.loc[y_train == 1, ["funded_amnt"] + PERFORMANCE_VARS].copy()

# Construimos la variable exposición
T["ead_obs"] = T["funded_amnt"] - T["princ_rec"]

# Recuperación post-impago: cobrado que no es principal, interés ni comisión.
T["rec_obs"] = (T["paym_rec_for_tot_amnt_fund"] - T["princ_rec"]
                - T["interest_rec"] - T["late_fees_rec"]).clip(lower=0)  # quita el ruido de -0.01

# EAD ≈ 0 -> LGD indefinido. Se excluyen (0,03%), no se les pone LGD = 1.
T = T[T["ead_obs"] > 1].copy()

print("% con recuperación > EAD:", (T["rec_obs"] > T["ead_obs"]).mean().round(4))
T["rec_obs"] = np.minimum(T["rec_obs"], T["ead_obs"])        # no se recupera más de lo expuesto

T["lgd_obs"] = 1 - T["rec_obs"] / T["ead_obs"]               # en [0, 1]
T["ccf_obs"] = (T["ead_obs"] / T["funded_amnt"]).clip(0, 1)  # EAD como proporción (para el modelo de EAD)
T["loss_obs"] = T["ead_obs"] * T["lgd_obs"]                  # pérdida en dinero

print(f"Filas: {len(T):,} | Pérdida total: {T['loss_obs'].sum():,.0f}")   # debe ser ≈ 2.224 M
print(T[["ead_obs", "lgd_obs", "ccf_obs", "loss_obs"]].describe().round(3).T)


% con recuperación > EAD: 0.0034
Filas: 223,695 | Pérdida total: 2,224,127,280
             count       mean       std   min       25%       50%        75%  \
ead_obs   223695.0  11019.656  7717.755  1.19  4892.610  9353.700  15383.905   
lgd_obs   223695.0      0.902     0.130  0.00     0.859     0.903      1.000   
ccf_obs   223695.0      0.690     0.222  0.00     0.556     0.739      0.868   
loss_obs  223695.0   9942.678  7157.040  0.00  4306.915  8392.050  13931.685   

              max  
ead_obs   40000.0  
lgd_obs       1.0  
ccf_obs       1.0  
loss_obs  40000.0  


### Objetivos observados de LGD y EAD (impagados del train)

**Muestra.** 223.695 impagados del train (13,1%); se excluyen 78 con EAD ≤ 1, donde el LGD es indefinido.

**Resultados.**

| Variable | Media | Desv. | Mediana | Comentario |
|---|---|---|---|---|
| EAD (`funded − princ_rec`) | 11.020 | 7.718 | 9.354 | Coincide con el principal pendiente donde éste se conserva |
| LGD | 0,902 | 0,130 | 0,903 | 39,7% de pérdida total; 38,3% sin recuperación |
| CCF (EAD / prestado) | 0,690 | 0,222 | 0,739 | Más dispersión que el LGD |
| Pérdida (EAD × LGD) | 9.943 | 7.157 | 8.392 | Suma train: 2.224 M |

**Señal exploratoria.** La mayor correlación de Spearman con el LGD es la fecha de emisión (+0,32); el resto de variables destacadas son sustitutas de la antigüedad del préstamo. El resto de señal es débil (|ρ| < 0,21).

**Lectura.** El LGD tiene poca variación y casi toda su señal es temporal, probablemente por madurez de los datos (los impagos recientes tienen menos recuperación observada). El EAD concentra la mayor parte de la variabilidad de la pérdida en dinero.

In [10]:
# Guardamos las variables construidas para no volverlas a construir.
T[["ead_obs", "rec_obs", "lgd_obs", "ccf_obs", "loss_obs"]] \
    .to_parquet(PRED_DIR / "targets_train_defaults.parquet")


# ¿Hay señal en las variables de originación? (correlación de Spearman con el LGD)
excluir = [c for c in PERFORMANCE_VARS + ["pd_lgbm"] if c in X_train.columns]
cand = X_train.loc[T.index].select_dtypes("number").drop(columns=excluir)
corr = cand.corrwith(T["lgd_obs"], method="spearman")

print(corr.reindex(corr.abs().sort_values(ascending=False).index).head(10).round(3))

issue_date_year                                    0.318
flag_missing_vintage_block                        -0.260
flag_missing_mths_since_last_installment_acc_op   -0.250
num_rev_trades_op_in_24mths                        0.209
num_installment_acc_op_in_24mths                   0.181
num_rev_trades_op_in_12mths                        0.174
num_open_trades_in_6mths                           0.156
num_inq                                            0.154
num_installment_acc_op_in_12mths                   0.129
num_inq_in_12mths                                  0.074
dtype: float64


`issue_date_year` es la variable que guarda mas correlación  con nuestra variable objetivo `lgd_obs`.

In [11]:
# ¿El LGD sube porque los impagos recientes aún no recuperaron nada?
chk = T.join(X_train["issue_date_year"])

tabla_anio = chk.groupby("issue_date_year").agg(
    n=("lgd_obs", "size"),
    lgd_medio=("lgd_obs", "mean"),
    pct_sin_recuperar=("rec_obs", lambda s: (s <= 0.01).mean()),
    ccf_medio=("ccf_obs", "mean"),
).round(3)

print(tabla_anio)

                     n  lgd_medio  pct_sin_recuperar  ccf_medio
issue_date_year                                                
2007               110      0.917              0.018      0.609
2008               403      0.897              0.027      0.631
2009               596      0.908              0.010      0.641
2010              1391      0.907              0.004      0.639
2011              2609      0.907              0.012      0.650
2012              6963      0.887              0.023      0.628
2013             16808      0.869              0.087      0.625
2014             33222      0.872              0.267      0.632
2015             61617      0.884              0.321      0.659
2016             56606      0.908              0.417      0.691
2017             33369      0.944              0.612      0.792
2018             10001      0.990              0.929      0.903


## Modelado para LGD

In [12]:
# Features: originación. Fuera: grade, desempeño, pd_lgbm y los log_ (los árboles no los necesitan)
TARGET_COLS = [
    "ead_obs",
    "rec_obs",
    "lgd_obs",
    "ccf_obs",
    "loss_obs",
]

FEATURES_LGD = [
    c for c in X_train.columns 
    if c not in PERFORMANCE_VARS + ["pd_lgbm", "grade"] + TARGET_COLS
    and not c.startswith("log_")
]


CAT_L = [c for c in FEATURES_LGD if not pd.api.types.is_numeric_dtype(X_train[c])]

XL = X_train.loc[T.index, FEATURES_LGD].copy()

CATS_L = {c: pd.Categorical(XL[c]).categories for c in CAT_L}    # categorías SOLO de train


for c in CAT_L:
    XL[c] = pd.Categorical(XL[c], categories=CATS_L[c])

yL = T["lgd_obs"].to_numpy()
eadL = T["ead_obs"].to_numpy()


def metricas_lgd(y, p, ead, nombre):
    p = np.clip(p, 0, 1)
    return {
        "modelo": nombre,
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "R2": r2_score(y, p),
        # En dinero: ¿la pérdida predicha (con el EAD REAL) iguala la real? 1 = exacto
        "Sum(pred·EAD)/Sum(real·EAD)": np.sum(p * ead) / np.sum(y * ead),
    }

# OOF sobre los impagados de train (KFold normal: el objetivo es continuo)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)


In [13]:
def oof(modelo, X):
    pred = np.zeros(len(yL))
    for tr, va in kf.split(X):
        m = clone(modelo).fit(X.iloc[tr], yL[tr])
        pred[va] = m.predict(X.iloc[va])
    return pred

# Modelo 1: constante (benchmark mínimo)
p_const = oof(DummyRegressor(strategy="mean"), XL)

# Modelo 2: LightGBM. 'cross_entropy' admite objetivos continuos en [0, 1] y mantiene las predicciones acotadas
lgbm_lgd = LGBMRegressor(
    objective="regression", 
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=200,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    random_state=SEED, 
    n_jobs=-1,
     verbosity=-1,
)
p_lgbm = oof(lgbm_lgd, XL)

tabla_lgd = pd.DataFrame([
    metricas_lgd(yL, p_const, eadL, "Constante"),
    metricas_lgd(yL, p_lgbm, eadL, "LightGBM"),
]).set_index("modelo").round(4)

display(tabla_lgd)

,MAE,RMSE,R2,Sum(pred·EAD)/Sum(real·EAD)
modelo,,,,
Constante,0.0882,0.1301,-0.0000,0.9994
LightGBM,0.0817,0.1251,0.0746,1.0030


In [14]:
# LGBM con solo la fecha de emisión: si R² ≈ 0,07, el resto de variables casi no aporta
p_solo_anio = oof(clone(lgbm_lgd), XL[["issue_date_year"]])
print(pd.DataFrame([metricas_lgd(yL, p_solo_anio, eadL, "Solo issue_date_year")]).round(4))

                 modelo     MAE    RMSE      R2  Sum(pred·EAD)/Sum(real·EAD)
0  Solo issue_date_year  0.0828  0.1264  0.0549                       1.0029


### Modelo de LGD (impagados del train, 5-fold OOF)

**Qué se hizo.** Se compararon dos modelos sobre los 223.695 impagados del train con KFold de 5 pliegues y `SEED`: una constante (media) y un LightGBM con objetivo `cross_entropy` (admite objetivos continuos en [0, 1]). Variables: originación, sin `grade`, sin variables de desempeño ni PD.

**Por qué.** El LGD se asume difícil de predecir, así que se exige a cada modelo que mejore al anterior. La métrica en dinero `Σ(pred·EAD) / Σ(real·EAD)` mide si la pérdida total cuadra, que es lo que importa para el ECL.

**Resultados.**

| Modelo | MAE | RMSE | R² | Σ(pred·EAD)/Σ(real·EAD) |
|---|---|---|---|---|
| Constante | 0,0882 | 0,1301 | 0,000 | 0,9994 |
| LightGBM | 0,0816 | 0,1251 | 0,075 | 1,0031 |

**Lectura.**
- LightGBM mejora el MAE un 7,5% y alcanza R² ≈ 0,075, en línea con lo habitual en LGD.
- La señal principal es el año de emisión: el LGD observado sube de ≈ 0,87 (2013-2015) a 0,99 (2018) porque los impagos recientes aún no han recuperado (92,9% de los de 2018 sin recuperación). Es un efecto de madurez de los datos.
- El mismo truncamiento afecta al CCF (0,63 → 0,90), porque los préstamos recientes que ya impagaron lo hicieron pronto.
- El test comparte el reparto temporal del train (split aleatorio), así que aprender este patrón es válido en el reto; no lo sería en un split temporal.

**Limitaciones.** Sin tuning de hiperparámetros; poca señal (R² < 0,10); el margen conservador sobre el LGD se aplica después, en el ECL final.

In [15]:
# 1) Modelo final: todos los impagados del train
lgd_final = clone(lgbm_lgd).fit(XL, yL)

# Métricas del modelo final sobre el train (in-sample)
p_final_train = np.clip(lgd_final.predict(XL), 0, 1)

tabla_lgd_final = pd.DataFrame([
    metricas_lgd(yL, p_lgbm, eadL, "LightGBM OOF"),
    metricas_lgd(yL, p_final_train, eadL, "LightGBM final (train)"),
]).set_index("modelo").round(4)

display(tabla_lgd_final)

def preparar(X):
    """Mismas columnas y categorías que en el entrenamiento (categorías definidas solo con train)."""
    Z = X[FEATURES_LGD].copy()
    for c in CAT_L:
        Z[c] = pd.Categorical(Z[c], categories=CATS_L[c])
    return Z

# 2) LGD para TODOS los préstamos del train
lgd_train = pd.Series(np.nan, index=X_train.index, name="lgd_pred")
es_imp = (np.asarray(y_train) == 1)

# Impagados: OOF (excluyendo los 78 que quedaron fuera por EAD <= 1)
lgd_train.loc[T.index] = np.clip(p_lgbm, 0, 1)

# Resto (no impagados + los 78 excluidos): modelo final
resto = lgd_train.index[lgd_train.isna()]
lgd_train.loc[resto] = np.clip(lgd_final.predict(preparar(X_train.loc[resto])), 0, 1)

# 3) LGD para el test (modelo final)
lgd_test = pd.Series(np.clip(lgd_final.predict(preparar(X_test)), 0, 1),
                     index=X_test.index, name="lgd_pred")

# 4) Comprobaciones y guardado
print("Train:", len(lgd_train), "| nulos:", lgd_train.isna().sum(), "| media:", lgd_train.mean().round(4))
print("Test: ", len(lgd_test),  "| nulos:", lgd_test.isna().sum(),  "| media:", lgd_test.mean().round(4))

# lgd_train.to_frame().to_parquet(PRED_DIR / "lgd_train.parquet")
# lgd_test.to_frame().to_parquet(PRED_DIR / "lgd_test.parquet")


,MAE,RMSE,R2,Sum(pred·EAD)/Sum(real·EAD)
modelo,,,,
LightGBM OOF,0.0817,0.1251,0.0746,1.003
LightGBM final (train),0.0798,0.1222,0.1166,1.003


Train: 1711714 | nulos: 0 | media: 0.9205
Test:  427929 | nulos: 0 | media: 0.9205


In [16]:
# assign las alinea POR ÍNDICE (no por posición). Antes, comprobamos que son compatibles.
assert X_train.index.is_unique and X_test.index.is_unique
assert lgd_train.index.equals(X_train.index), "El índice del LGD de train no coincide"
assert lgd_test.index.equals(X_test.index), "El índice del LGD de test no coincide"

X_train = X_train.assign(lgd_pred=lgd_train)   # impagados: OOF | resto: modelo final
X_test = X_test.assign(lgd_pred=lgd_test)      # modelo final

# Guardar (mismos archivos: ahora llevan PD y LGD)
X_train.to_parquet(PRED_DIR / "X_train_lgd.parquet")
X_test.to_parquet(PRED_DIR / "X_test_lgd.parquet")

# Comprobaciones
chk_tr = pd.read_parquet(PRED_DIR / "X_train_lgd.parquet")
chk_te = pd.read_parquet(PRED_DIR / "X_test_lgd.parquet")
print(chk_tr.shape, chk_te.shape)                    # una columna más: 49
print("Nulos lgd_pred (train/test):", chk_tr["lgd_pred"].isna().sum(), chk_te["lgd_pred"].isna().sum())
print("LGD media train/test:", chk_tr["lgd_pred"].mean().round(4), chk_te["lgd_pred"].mean().round(4))
print("LGD entre 0 y 1:", chk_tr["lgd_pred"].between(0, 1).all(), chk_te["lgd_pred"].between(0, 1).all())
print("Columnas de predicción:", [c for c in chk_tr.columns if c in ("pd_lgbm", "lgd_pred")])

(1711714, 58) (427929, 58)
Nulos lgd_pred (train/test): 0 0
LGD media train/test: 0.9205 0.9205
LGD entre 0 y 1: True True
Columnas de predicción: ['pd_lgbm', 'lgd_pred']


### EAD

In [17]:
LGD = ["lgd_pred"]
PRED_COLS = PD + LGD + ["ccf_pred", "ead_pred"]    # salidas de modelos: nunca son entradas
NUM_BASE = [c for c in NUM if c not in LOG_VARS + FLAG_VARS + PERFORMANCE_VARS + PRED_COLS]

assert not set(PRED_COLS) & set(NUM_BASE)
print(len(NUM_BASE))    # debe volver a 32 (29 originales + 3 de EDA), no 34

32


In [18]:
T = pd.read_parquet(PRED_DIR / "targets_train_defaults.parquet")
assert (y_train[T.index] == 1).all()
ead_chk = X_train.loc[T.index, "funded_amnt"] - X_train.loc[T.index, "princ_rec"]
assert np.allclose(ead_chk, T["ead_obs"]), "Las filas no están alineadas con X_train"
print("Impagados:", len(T), "| CCF medio:", T["ccf_obs"].mean().round(3))

Impagados: 223695 | CCF medio: 0.69


In [19]:
# Variables: originación. Fuera: grade (ya no está en CAT), desempeño, predicciones y log_
FEATURES_EAD = NUM_BASE + FLAG_VARS + CAT

# Categorías tomadas de TODO el train (no solo de impagados), para predecir a cualquier préstamo
CATS = {c: pd.Categorical(X_train[c]).categories for c in CAT}

def para_lgbm(X):
    Z = X[FEATURES_EAD].copy()
    for c in CAT:
        Z[c] = pd.Categorical(Z[c], categories=CATS[c])
    return Z

XE = para_lgbm(X_train.loc[T.index])                 # para LightGBM
XR = X_train.loc[T.index, FEATURES_EAD]              # para la regresión (texto crudo)
yE = T["ccf_obs"].to_numpy()
fundE = X_train.loc[T.index, "funded_amnt"].to_numpy()

# Modelo lineal: sin variables colineales, pero CON issue_date_year
RIDGE_DROP = ["earliest_cr_line_year", "monthly_payment"]
num_lin = [c for c in NUM_BASE if c not in RIDGE_DROP]

ridge = Pipeline([
    ("prep", ColumnTransformer([
    ("num", StandardScaler(), num_lin),
    ("flag", "passthrough", FLAG_VARS),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=0.005), CAT),
])),
    ("model", Ridge(alpha=10.0, random_state=SEED)),
])

# LightGBM: 'cross_entropy' admite objetivos continuos en [0, 1]
lgbm_ead = LGBMRegressor(
    objective="cross_entropy",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=200,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def oof(modelo, X):
    """Cada impagado se predice con un modelo que no lo vio."""
    pred = np.zeros(len(yE))
    for tr, va in kf.split(X):
        m = clone(modelo).fit(X.iloc[tr], yE[tr])
        pred[va] = m.predict(X.iloc[va])
    return np.clip(pred, 0, 1)

def metricas_ead(y, p, fund, nombre):
    ead_real, ead_pred = y * fund, np.clip(p, 0, 1) * fund
    return {
        "modelo": nombre,
        "MAE_ccf": mean_absolute_error(y, p),
        "R2_ccf": r2_score(y, p),
        "MAE_$": mean_absolute_error(ead_real, ead_pred),       # error medio por préstamo, en dinero
        "Sum(pred)/Sum(real)": ead_pred.sum() / ead_real.sum(), # ¿cuadra el total?
    }

p_const = oof(DummyRegressor(strategy="mean"), XR)
p_ridge = oof(ridge, XR)
p_lgbm_ead = oof(lgbm_ead, XE)

tabla_ead = pd.DataFrame([
    metricas_ead(yE, p_const, fundE, "Constante"),
    metricas_ead(yE, p_ridge, fundE, "Ridge"),
    metricas_ead(yE, p_lgbm_ead, fundE, "LightGBM"),
]).set_index("modelo").round(4)
display(tabla_ead)

,MAE_ccf,R2_ccf,MAE_$,Sum(pred)/Sum(real)
modelo,,,,
Constante,0.1806,-0.0000,2748.0462,0.9715
Ridge,0.1545,0.2211,2254.0889,1.0000
LightGBM,0.1491,0.2632,2176.8117,1.0001


In [20]:
# Modelo final con todos los impagados del train
ead_final = clone(lgbm_ead).fit(XE, yE)

# Train: impagados -> OOF | resto (no impagados + 78 excluidos) -> modelo final
ccf_train = pd.Series(np.nan, index=X_train.index, name="ccf_pred")
ccf_train.loc[T.index] = p_lgbm_ead
resto = ccf_train.index[ccf_train.isna()]
ccf_train.loc[resto] = np.clip(ead_final.predict(para_lgbm(X_train.loc[resto])), 0, 1)

# Test: modelo final
ccf_test = pd.Series(np.clip(ead_final.predict(para_lgbm(X_test)), 0, 1),
                     index=X_test.index, name="ccf_pred")

# Añadir al dataframe (por índice) junto con el EAD en dinero
X_train_e = X_train.assign(ccf_pred=ccf_train)
X_test_e = X_test.assign(ccf_pred=ccf_test)

for D_ in (X_train_e, X_test_e):
    D_["ead_pred"] = D_["ccf_pred"] * D_["funded_amnt"]

X_train_e.to_parquet(PRED_DIR / "X_train_ead.parquet")
X_test_e.to_parquet(PRED_DIR / "X_test_ead.parquet")

print(X_train_e.shape, X_test_e.shape)                       # dos columnas más que X_train_lgd
print("Nulos:", X_train_e[["ccf_pred", "ead_pred"]].isna().sum().sum(),
      X_test_e[["ccf_pred", "ead_pred"]].isna().sum().sum())
print("CCF medio train/test:", X_train_e["ccf_pred"].mean().round(3), X_test_e["ccf_pred"].mean().round(3))

(1711714, 60) (427929, 60)
Nulos: 0 0
CCF medio train/test: 0.706 0.706


In [21]:
print(set(X_train.columns) - set(X_test.columns))   # debe ser exactamente PERFORMANCE_VARS

set()


In [22]:
COLS = ["funded_amnt", "pd_lgbm", "lgd_pred", "ccf_pred", "ead_pred"]

Xtr = pd.read_parquet(PRED_DIR / "X_train_ead.parquet", columns=COLS)
Xte = pd.read_parquet(PRED_DIR / "X_test_ead.parquet", columns=COLS)
T = pd.read_parquet(PRED_DIR / "targets_train_defaults.parquet")

In [23]:
# ECL estimado (sin margen) y ECL real de train: pérdida observada si impagó, 0 si no
ecl_base = Xtr["pd_lgbm"] * Xtr["lgd_pred"] * Xtr["ead_pred"]
ecl_real = pd.Series(0.0, index=Xtr.index)
ecl_real.loc[T.index] = T["loss_obs"]            # los 78 excluidos (EAD<=1) quedan en 0: efecto despreciable

def evaluar(est, real, nombre, limite=0.10):
    est, real = np.asarray(est), np.asarray(real)
    imp = real > 0
    return {
        "estrategia": nombre,
        "ECL_est/ECL_real": est.sum() / real.sum(),              # restricción 1: debe ser > 1
        "%_cartera_por_debajo": (est < real).mean() * 100,       # restricción 2: debe ser <= 10
        "MAE": np.abs(est - real).mean(),
        "MAPE_impagados_%": (np.abs(est - real)[imp] / real[imp]).mean() * 100,
        "cumple": bool(est.sum() > real.sum() and (est < real).mean() <= limite),
    }

print(pd.DataFrame([evaluar(ecl_base, ecl_real, "Sin margen")]).round(3).T)

                                         0
estrategia                      Sin margen
ECL_est/ECL_real                     1.002
%_cartera_por_debajo                12.805
MAE                               2004.229
MAPE_impagados_%      2060503259501.456055
cumple                               False


In [24]:
OBJ = 0.095                                       # objetivo en train (deja 0,5 pt de colchón)
real = ecl_real.to_numpy()
base = ecl_base.to_numpy()
pdv = Xtr["pd_lgbm"].to_numpy()
lgd_ead = (Xtr["lgd_pred"] * Xtr["ead_pred"]).to_numpy()    # pérdida si impaga (sin multiplicar por PD)
imp = real > 0
N = len(real)
permitidos = int(OBJ * N)                         # préstamos que pueden quedar por debajo del real

# --- A) factor único k: se deja justo 'permitidos' impagados con real > k * base
r = np.sort(real[imp] / base[imp])[::-1]
k = r[permitidos]
est_A = k * base

# --- B) tramo de PD alta: se recorre q y se calcula el c mínimo que cumple
filas = []
for q in np.arange(0.02, 0.40, 0.01):
    umbral = np.quantile(pdv, 1 - q)
    top = pdv >= umbral
    fuera = int((imp & ~top & (base < real)).sum())     # impagados fuera del tramo (quedan por debajo)
    holgura = permitidos - fuera
    if holgura < 0:
        continue                                        # tramo demasiado pequeño: no alcanza
    s = np.sort(real[imp & top] / lgd_ead[imp & top])[::-1]
    c = s[holgura] if holgura < len(s) else 0.0
    est = np.where(top, np.maximum(c * lgd_ead, base), base)   # nunca por debajo del ECL base
    filas.append({"q_top": round(q, 2), "umbral_PD": umbral, "c": c,
                  "ECL_est/ECL_real": est.sum() / real.sum()})

tab_B = pd.DataFrame(filas).sort_values("ECL_est/ECL_real")
print(tab_B.head(5).round(3))

mejor = tab_B.iloc[0]
umbral_B, c_B = mejor["umbral_PD"], mejor["c"]
est_B = np.where(pdv >= umbral_B, np.maximum(c_B * lgd_ead, base), base)

print(f"k (estrategia A) = {k:.2f} | tramo B: q={mejor['q_top']}, c={c_B:.2f}")
display(pd.DataFrame([
    evaluar(ecl_base, ecl_real, "Sin margen"),
    evaluar(est_A, ecl_real, "A: factor único k"),
    evaluar(est_B, ecl_real, "B: tramo de PD alta"),
]).set_index("estrategia").round(3))

   q_top  umbral_PD      c  ECL_est/ECL_real
1   0.11      0.271  1.299             1.952
0   0.10      0.280  1.405             1.967
2   0.12      0.263  1.244             1.974
3   0.13      0.255  1.206             2.007
4   0.14      0.248  1.177             2.044
k (estrategia A) = 3.20 | tramo B: q=0.11, c=1.30


,ECL_est/ECL_real,%_cartera_por_debajo,MAE,MAPE_impagados_%,cumple
estrategia,,,,,
Sin margen,1.002,12.805,2004.229,2.060503e+12,False
A: factor único k,3.209,9.500,3789.963,6.601395e+12,True
B: tramo de PD alta,1.952,9.500,2651.437,2.060503e+12,True


In [25]:
real = ecl_real.to_numpy()
base = ecl_base.to_numpy()
pdv = Xtr["pd_lgbm"].to_numpy()
lgd_ead = (Xtr["lgd_pred"] * Xtr["ead_pred"]).to_numpy()    # pérdida si impaga (sin multiplicar por PD)
N = len(real)
OBJ = 0.095

def evaluar(est, real, nombre, limite=0.10, piso=1.0):
    """MAPE solo donde la pérdida real es apreciable (> 1 $): con ~0 en el denominador no tiene sentido."""
    est, real = np.asarray(est), np.asarray(real)
    m = real > piso
    ape = np.abs(est[m] - real[m]) / real[m]
    return {
        "estrategia": nombre,
        "ECL_est/ECL_real": est.sum() / real.sum(),            # restricción 1: > 1
        "%_cartera_por_debajo": (est < real).mean() * 100,     # restricción 2: <= 10
        "MAE": np.abs(est - real).mean(),
        "MAPE_%": ape.mean() * 100,
        "MdAPE_%": np.median(ape) * 100,
        "cumple": bool(est.sum() > real.sum() and (est < real).mean() <= limite),
    }

def ajustar(score, real, base, lgd_ead, objetivo, qs=np.arange(0.02, 0.40, 0.01)):
    """Elige el tramo (los q% con mayor 'score') y el factor c que dejan 'objetivo' de cartera
    por debajo del real con el menor ECL total. Devuelve los parámetros a congelar."""
    imp = real > 0
    permitidos = int(objetivo * len(real))
    mejor = None
    for q in qs:
        umbral = np.quantile(score, 1 - q)
        top = score >= umbral
        # impagados fuera del tramo: se quedan en su ECL base, y casi todos por debajo del real
        holgura = permitidos - int((imp & ~top & (base < real)).sum())
        if holgura < 0:
            continue                                           # tramo demasiado pequeño
        s = np.sort(real[imp & top] / lgd_ead[imp & top])[::-1]
        c = s[holgura] if holgura < len(s) else 0.0
        total = np.where(top, np.maximum(c * lgd_ead, base), base).sum()
        if mejor is None or total < mejor["total"]:
            mejor = {"q": q, "umbral": umbral, "c": c, "total": total}
    return mejor

def aplicar(score, base, lgd_ead, m):
    """Aplica parámetros ya calibrados (el umbral es un VALOR, no un percentil)."""
    return np.where(score >= m["umbral"], np.maximum(m["c"] * lgd_ead, base), base)

SCORES = {
    "B: por PD": pdv,
    "B2: por PD / coste": pdv / np.maximum(lgd_ead, 1.0),
}

# 1) Comparativa corregida
filas, params = [evaluar(base, real, "Sin margen")], {}
for nombre, sc in SCORES.items():
    params[nombre] = ajustar(sc, real, base, lgd_ead, OBJ)
    est = aplicar(sc, base, lgd_ead, params[nombre])
    filas.append({**evaluar(est, real, nombre),
                  "q_top": round(params[nombre]["q"], 2), "c": round(params[nombre]["c"], 2)})
display(pd.DataFrame(filas).set_index("estrategia").round(3))

# 2) ¿Se transfiere la calibración? Calibrar en una mitad del train, medir en la otra
print("\nCalibrado en A -> medido en B (dentro del train):")
for r in range(3):
    idx = np.random.default_rng(SEED + r).permutation(N)
    A, B = idx[: N // 2], idx[N // 2:]
    for nombre, sc in SCORES.items():
        m = ajustar(sc[A], real[A], base[A], lgd_ead[A], OBJ)
        est_B = aplicar(sc[B], base[B], lgd_ead[B], m)
        print(f"  r={r} {nombre:20s} % por debajo en B = {(est_B < real[B]).mean()*100:.3f} "
              f"| ECL/real en B = {est_B.sum()/real[B].sum():.3f}")

# 3) Coste del colchón (calibración con todo el train, tramo por PD)
print("\nCoste de cada décima de punto de colchón:")
for obj in [0.090, 0.095, 0.098, 0.100]:
    m = ajustar(pdv, real, base, lgd_ead, obj)
    print(f"  objetivo {obj:.3f} -> ECL/real = {m['total'] / real.sum():.3f}")

,ECL_est/ECL_real,%_cartera_por_debajo,MAE,MAPE_%,MdAPE_%,cumple,q_top,c
estrategia,,,,,,,,
Sin margen,1.002,12.805,2004.229,82.319,79.542,False,NaN,NaN
B: por PD,1.952,9.500,2651.437,83.942,80.988,True,0.11,1.30
B2: por PD / coste,1.617,9.500,2576.498,103.428,78.981,True,0.17,1.55



Calibrado en A -> medido en B (dentro del train):
  r=0 B: por PD            % por debajo en B = 9.463 | ECL/real en B = 1.959
  r=0 B2: por PD / coste   % por debajo en B = 9.492 | ECL/real en B = 1.618
  r=1 B: por PD            % por debajo en B = 9.479 | ECL/real en B = 1.954
  r=1 B2: por PD / coste   % por debajo en B = 9.509 | ECL/real en B = 1.613
  r=2 B: por PD            % por debajo en B = 9.618 | ECL/real en B = 1.926
  r=2 B2: por PD / coste   % por debajo en B = 9.608 | ECL/real en B = 1.589

Coste de cada décima de punto de colchón:
  objetivo 0.090 -> ECL/real = 2.143
  objetivo 0.095 -> ECL/real = 1.952
  objetivo 0.098 -> ECL/real = 1.849
  objetivo 0.100 -> ECL/real = 1.776


In [26]:
import numpy as np, pandas as pd, json
from pathlib import Path

SEED = 42                                       # SEED única (import desde config.py)
PRED_DIR = Path("../data/processed")
COLS = ["funded_amnt", "pd_lgbm", "lgd_pred", "ccf_pred", "ead_pred"]

Xtr = pd.read_parquet(PRED_DIR / "X_train_ead.parquet", columns=COLS)
Xte = pd.read_parquet(PRED_DIR / "X_test_ead.parquet", columns=COLS)
T = pd.read_parquet(PRED_DIR / "targets_train_defaults.parquet")

In [27]:
sc = SCORES["B2: por PD / coste"]
OBJ = 0.095
res = []
for r in range(20):
    idx = np.random.default_rng(SEED + r).permutation(N)
    cal, ev = idx[: int(0.8 * N)], idx[int(0.8 * N):]
    m = ajustar(sc[cal], real[cal], base[cal], lgd_ead[cal], OBJ)
    est = aplicar(sc[ev], base[ev], lgd_ead[ev], m)
    res.append(((est < real[ev]).mean() * 100, est.sum() / real[ev].sum()))
res = np.array(res)

desv = res[:, 0] - OBJ * 100                    # desviación respecto al objetivo, en puntos
print(f"% por debajo: media {res[:,0].mean():.3f} | sd {res[:,0].std():.3f} | máx {res[:,0].max():.3f}")
print(f"ECL/real:     media {res[:,1].mean():.3f}")

# Regla práctica: colchón = desviación media + 3 sd + 0,10 pt extra por el cambio OOF -> modelo final
colchon = desv.mean() + 3 * desv.std() + 0.10
OBJ_FINAL = (10 - colchon) / 100
print(f"Colchón sugerido: {colchon:.2f} pt -> OBJ_FINAL = {OBJ_FINAL:.4f}")

% por debajo: media 9.475 | sd 0.043 | máx 9.543
ECL/real:     media 1.620
Colchón sugerido: 0.20 pt -> OBJ_FINAL = 0.0980


In [28]:
def score_b2(p, le):                                  # mismo score que en la comparativa
    return p / np.maximum(le, 1.0)

params = ajustar(score_b2(pdv, lgd_ead), real, base, lgd_ead, OBJ_FINAL)
params = {k: float(v) for k, v in params.items()}      # umbral, c, q: ya no se tocan
json.dump(params, open(PRED_DIR / "params_margen.json", "w"))

est_train = aplicar(score_b2(pdv, lgd_ead), base, lgd_ead, params)
print(params)
display(pd.DataFrame([evaluar(est_train, real, "B2 final (train)")]).set_index("estrategia").round(3))

{'q': 0.15999999999999998, 'umbral': 3.959690607086646e-05, 'c': 1.4882615849686895, 'total': 3414416469.227949}


,ECL_est/ECL_real,%_cartera_por_debajo,MAE,MAPE_%,MdAPE_%,cumple
estrategia,,,,,,
B2 final (train),1.535,9.796,2491.904,100.181,78.636,True


In [30]:
# ===== CELDA FINAL: aplicar la estrategia CONGELADA al test y evaluar =====
# Las variables de desempeño del test están en el propio X_test_ead.parquet (mismas filas que Xte)
P = pd.read_parquet(PRED_DIR / "X_test_ead.parquet", columns=["funded_amnt"] + PERFORMANCE_VARS)
y_te = pd.read_parquet(PRED_DIR / "y_test.parquet")["y"].to_numpy()
assert len(P) == len(Xte) == len(y_te), "Filas desalineadas"

# Pérdida real del test: MISMAS fórmulas que en train (solo para evaluar, nunca para decidir)
ead_obs = P["funded_amnt"] - P["princ_rec"]
rec_obs = (P["paym_rec_for_tot_amnt_fund"] - P["princ_rec"]
           - P["interest_rec"] - P["late_fees_rec"]).clip(lower=0)
perdida = (ead_obs - np.minimum(rec_obs, ead_obs)).clip(lower=0)       # = EAD × LGD observados
real_te = np.where((y_te == 1) & (ead_obs > 1), perdida, 0.0)

# ECL estimado: parámetros calibrados SOLO con train (params_margen.json)
pd_te = Xte["pd_lgbm"].to_numpy()
le_te = (Xte["lgd_pred"] * Xte["ead_pred"]).to_numpy()
est_te = aplicar(score_b2(pd_te, le_te), pd_te * le_te, le_te, params)

r = evaluar(est_te, real_te, "B2 (test)")
print("=========== RESULTADO FINAL ===========")
print(f"1) ECL total estimado:  {est_te.sum():,.0f}")
print(f"2) MAE de ECL_j: {r['MAE']:,.2f} | MAPE de ECL_j: {r['MAPE_%']:.1f}% (sobre pérdida real > 1)")
print("--------------- control ---------------")
print(f"ECL real (test): {real_te.sum():,.0f} | ratio est/real: {r['ECL_est/ECL_real']:.3f}")
print(f"% cartera por debajo del real: {r['%_cartera_por_debajo']:.2f}% (límite 10%) | cumple: {r['cumple']}")
print(f"Sanidad: tasa de impago test {y_te.mean():.4f} | pérdida media por impagado {real_te[y_te == 1].mean():,.0f}")

=========== RESULTADO FINAL ===========
1) ECL total estimado:  853,224,311
2) MAE de ECL_j: 2,485.40 | MAPE de ECL_j: 91.5% (sobre pérdida real > 1)
--------------- control ---------------
ECL real (test): 555,337,850 | ratio est/real: 1.536
% cartera por debajo del real: 9.81% (límite 10%) | cumple: True
Sanidad: tasa de impago test 0.1307 | pérdida media por impagado 9,927
